In [1]:
import numpy as np
import seaborn as sns
import os
from scipy.stats import kurtosis, skew
import pandas as pd
import pickle

from rcv_distribution import *
from MDS_analysis import *
from voting_rules import *
from consistency import *
from null_elections import *

c:\Users\mahsh\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\mahsh\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (
c:\Users\mahsh\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.0
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Dropping trucated ballots from the real eletions:
1. Only bullet votes
2. All voluntarily truncation 

In [2]:
# Building the Dataframe

directory = "dataverse_files_2025"
election_table = pd.read_csv("election_table.csv")

# Dataframe: truncation analysis
trunc = pd.DataFrame(columns = ["filename", "candidates", "choices", 
                                "gamma", "og_irv", "og_condorcet", 
                                "og_plurality", "Bvote_free_irv", 
                                "Bvote_free_condorcet", "Bvote_free_plurality",
                                "Tvote_free_irv", "Tvote_free_condorcet", "Tvote_free_plurality",
                                "bullet_votes", "voluntarily_truncated_votes", "total_votes"
                                ])


for filename in os.listdir(directory):
    if filename in election_table["filename"].values:
        candidates = election_table.loc[election_table["filename"]==filename, "candidates"].values[0]
        if candidates > 2:
            trunc.loc[len(trunc)] = [pd.NA] * len(trunc.columns)
            
            i = len(trunc) - 1
            trunc.at[i, "filename"] = filename

            
            trunc.at[i, "candidates"] = candidates
            trunc.at[i, "choices"] = election_table.loc[election_table["filename"]==filename, "choices"].values[0]
            trunc.at[i, "gamma"] = election_table.loc[election_table["filename"]==filename, "gamma"].values[0]
            

            trunc.at[i, "og_irv"]  = election_table.loc[election_table["filename"]==filename, "irv_winner"].values[0]
            trunc.at[i, "og_condorcet"]  = election_table.loc[election_table["filename"]==filename, "condorcet_winner"].values[0]
            trunc.at[i, "og_plurality"]  = election_table.loc[election_table["filename"]==filename, "plurality_winner"].values[0]



In [3]:
trunc

,filename,candidates,choices,gamma,og_irv,og_condorcet,og_plurality,Bvote_free_irv,Bvote_free_condorcet,Bvote_free_plurality,Tvote_free_irv,Tvote_free_condorcet,Tvote_free_plurality,bullet_votes,voluntarily_truncated_votes,total_votes
0,Alaska_04102020_PRESIDENTOFTHEUNITEDSTATES.csv,8,5,0.670398,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,Alaska_08162022_HouseofRepresentativesSpecial.csv,3,4,0.966842,"Peltola, Mary S.","Begich, Nick","Peltola, Mary S.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Alaska_11052024_President.csv,8,8,0.833383,Trump/Vance,Trump/Vance,Trump/Vance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Alaska_11052024_StateHouseD1.csv,3,4,0.953816,"Bynum, Jeremy T.","Bynum, Jeremy T.","Bynum, Jeremy T.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Alaska_11052024_StateHouseD15.csv,3,4,0.98254,"Costello, Mia","Costello, Mia","Costello, Mia",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,Vineyard_11022021_Mayor.csv,3,3,0.800911,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
357,Vineyard_11052019_CityCouncil.csv,7,7,0.237833,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
358,Westbrook_11052024_Mayor.csv,3,3,0.890167,"Morse, David","Morse, David","Morse, David",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
359,WoodlandHills_11022021_Mayor.csv,3,3,0.862595,BRENT WINDER,BRENT WINDER,BRENT WINDER,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# --- prepare data from `trunc` ---
trunc["gamma"] = pd.to_numeric(trunc["gamma"], errors="coerce")
trunc["candidates"] = pd.to_numeric(trunc["candidates"], errors="coerce")

dfv = trunc.loc[trunc["gamma"].notna() & trunc["candidates"].notna(), ["level", "candidates", "gamma"]].copy()
dfv["one_minus_gamma"] = 1 - dfv["gamma"]

# (same filter as before) keep only elections with < 16 candidates
dfv = dfv[dfv["candidates"] < 16]

# aggregate (mean of 1 - gamma) by level
order_raw   = ["LOCAL", "STATE", "FEDERAL"]
order_labels= ["Local", "State", "Federal"]

agg = (
    dfv.groupby("level", as_index=False)["one_minus_gamma"]
       .mean()
       .set_index("level")
       .reindex(order_raw)   # keep desired order; may produce NaN if a level is missing
       .dropna()             # drop levels not present
       .reset_index()
)

levels = [order_labels[order_raw.index(lv)] for lv in agg["level"]]
vals   = (agg["one_minus_gamma"] * 100).to_list()  # percent scale

x = list(range(len(levels)))  # 0,1,...

fig, ax = plt.subplots(figsize=(6.5, 4.2), dpi=150)

# single series plotted like your format
ax.scatter(x, vals, s=60, marker='o', label='1 − gamma (mean)', zorder=3)
ax.plot(x, vals, lw=0.8, alpha=0.6, zorder=2, solid_capstyle='round')

# ticks/axes
ax.set_xticks(x, levels)
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.set_ylim(0, 100)
ax.grid(axis='y', linestyle=':', linewidth=0.8, alpha=0.6)

ax.set_xlabel("Level")
ax.set_ylabel("1 − gamma")

# annotate values above points
for xi, y in zip(x, vals):
    ax.annotate(f'{y:.0f}%', (xi, y), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=9)

plt.title("1 − gamma by level (candidates < 16)")
ax.legend(frameon=False, loc='best')
plt.tight_layout()
# plt.savefig('one_minus_gamma_by_level_lt16.png', dpi=300)

plt.show()


KeyError: "['level'] not in index"

In [5]:
def load_data(filename, save_folder):
    # Extract the base filename without extension
    base_filename = os.path.splitext(os.path.basename(filename))[0]

    # Create the save paths for dictionary and list
    dict_load_path = os.path.join(save_folder, f"{base_filename}_ballots.pkl")
    list_load_path = os.path.join(save_folder, f"{base_filename}_candidates.pkl")

    # Load the dictionary
    with open(dict_load_path, 'rb') as dict_file:
        data_dict = pickle.load(dict_file)

    # Load the list
    with open(list_load_path, 'rb') as list_file:
        data_list = pickle.load(list_file)

    return data_dict, data_list

In [9]:
# dropping the only bullet_vote  ballots and recalculating winners 
saved = "saved_ballots_and_candidates"
total_change = 0
total = 0
Bvotes_average = 0
for filename in os.listdir(directory):
    if filename in trunc["filename"].values:
        base_filename = filename[:-4]
        try:
            ballots, candidates = load_data(base_filename, saved)
        except Exception as e:
            print(filename, " ", e)

        altered_ballots = {}
        Bvotes = 0
        total_votes = 0
        for b in ballots:
            if len(b) > 0:
                total_votes += ballots[b]
            
            if len(b) > 1:
                altered_ballots[b] = ballots[b]
            elif len(b) == 1:
                Bvotes += ballots[b]

        Bvotes_average += (Bvotes / total_votes)
            
        election = voting_rules(altered_ballots, candidates)
        Bvote_free_irv = election.irv()[0]
        # print(Bvote_free_irv, " ", trunc.loc[trunc["filename"] == filename, "og_irv"].values[0], " ", Bvotes, "/", total_votes)
        trunc.loc[trunc["filename"]==filename, "Bvote_free_irv"] = Bvote_free_irv

        Bvote_free_condorcet = election.condorcet()
        trunc.loc[trunc["filename"]==filename, "Bvote_free_condorcet"] = Bvote_free_condorcet

        Bvote_free_plurality = election.plurality()
        trunc.loc[trunc["filename"]==filename, "Bvote_free_plurality"] = Bvote_free_plurality 

        trunc.loc[trunc["filename"]==filename, "bullet_votes"] = Bvotes
        trunc.loc[trunc["filename"]==filename, "total_votes"] = total_votes



        if (Bvote_free_irv != trunc.loc[trunc["filename"]==filename, "og_irv"].values[0]):
            total_change += 1
        total += 1
        
print(total_change, "/", total)
print(Bvotes_average/total)





53 / 361
0.3430334387926418


In [69]:
# dropping truncated ballots

saved = "saved_ballots_and_candidates"
total_change = 0
total = 0
wierd = 0
for filename in os.listdir(directory):
    if filename in trunc["filename"].values:
        base_filename = filename[:-4]
        try:
            ballots, candidates = load_data(base_filename, saved)
        except Exception as e:
            print(filename, " ", e)

        choices = trunc.loc[trunc["filename"]==filename, "choices"].values[0]
        candidates_num = trunc.loc[trunc["filename"]==filename, "candidates"].values[0]

        if choices >= candidates_num: 
            altered_ballots = {}
            
            Tvotes = 0
            flag = False
            for b in ballots:
                if len(b) == candidates_num:
                    altered_ballots[b] = ballots[b]
                elif len(b) > 0 and len(b) < candidates_num:
                    Tvotes += ballots[b]
                elif len(b) > candidates_num:
                    flag = True
            
            if flag is True:
                wierd += 1
                print(filename, " is wierd")
            
            election = voting_rules(altered_ballots, candidates)
            Tvote_free_irv = election.irv()[0]
            # print(Bvote_free_irv, " ", trunc.loc[trunc["filename"] == filename, "og_irv"].values[0], " ", Bvotes, "/", total_votes)
            trunc.loc[trunc["filename"]==filename, "Tvote_free_irv"] = Tvote_free_irv

            Tvote_free_condorcet = election.condorcet()
            trunc.loc[trunc["filename"]==filename, "Tvote_free_condorcet"] = Tvote_free_condorcet

            Tvote_free_plurality = election.plurality()
            trunc.loc[trunc["filename"]==filename, "Tvote_free_plurality"] = Tvote_free_plurality 

            trunc.loc[trunc["filename"]==filename, "voluntarily_truncated_votes"] = Tvotes


            if (Tvote_free_irv != Tvote_free_condorcet):
                total_change += 1
            else:
                total += 1
        
        
print(total_change, "/", total)





5 / 188


In [70]:
trunc

,filename,candidates,choices,gamma,og_irv,og_condorcet,og_plurality,Bvote_free_irv,Bvote_free_condorcet,Bvote_free_plurality,Tvote_free_irv,Tvote_free_condorcet,Tvote_free_plurality,bullet_votes,voluntarily_truncated_votes,total_votes
0,Alaska_04102020_PRESIDENTOFTHEUNITEDSTATES.csv,8,5,0.670398,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,<NA>,<NA>,<NA>,3922,<NA>,19742
1,Alaska_08162022_HouseofRepresentativesSpecial.csv,3,4,0.966842,"Peltola, Mary S.","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.",56054,133333,188852
2,Alaska_11052024_President.csv,8,8,0.833383,Trump/Vance,Trump/Vance,Trump/Vance,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,228278,301747,338650
3,Alaska_11052024_StateHouseD1.csv,3,4,0.953816,"Bynum, Jeremy T.","Bynum, Jeremy T.","Bynum, Jeremy T.","Moran, Agnes C.","Moran, Agnes C.","Moran, Agnes C.","Echohawk, Grant","Moran, Agnes C.","Bynum, Jeremy T.",5310,6769,8163
4,Alaska_11052024_StateHouseD15.csv,3,4,0.98254,"Costello, Mia","Costello, Mia","Costello, Mia","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny",6237,7576,8820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,Vineyard_11022021_Mayor.csv,3,3,0.800911,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,317,385,1537
357,Vineyard_11052019_CityCouncil.csv,7,7,0.237833,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,64,452,1089
358,Westbrook_11052024_Mayor.csv,3,3,0.890167,"Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David",4464,5663,9560
359,WoodlandHills_11022021_Mayor.csv,3,3,0.862595,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,128,262,655


In [72]:
trunc ["level"] = pd.NA

In [76]:
trunc_copy = trunc.copy()
trunc_copy

,filename,candidates,choices,gamma,og_irv,og_condorcet,og_plurality,Bvote_free_irv,Bvote_free_condorcet,Bvote_free_plurality,Tvote_free_irv,Tvote_free_condorcet,Tvote_free_plurality,bullet_votes,voluntarily_truncated_votes,total_votes,level
0,Alaska_04102020_PRESIDENTOFTHEUNITEDSTATES.csv,8,5,0.670398,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,<NA>,<NA>,<NA>,3922,<NA>,19742,<NA>
1,Alaska_08162022_HouseofRepresentativesSpecial.csv,3,4,0.966842,"Peltola, Mary S.","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.",56054,133333,188852,<NA>
2,Alaska_11052024_President.csv,8,8,0.833383,Trump/Vance,Trump/Vance,Trump/Vance,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,228278,301747,338650,<NA>
3,Alaska_11052024_StateHouseD1.csv,3,4,0.953816,"Bynum, Jeremy T.","Bynum, Jeremy T.","Bynum, Jeremy T.","Moran, Agnes C.","Moran, Agnes C.","Moran, Agnes C.","Echohawk, Grant","Moran, Agnes C.","Bynum, Jeremy T.",5310,6769,8163,<NA>
4,Alaska_11052024_StateHouseD15.csv,3,4,0.98254,"Costello, Mia","Costello, Mia","Costello, Mia","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny",6237,7576,8820,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,Vineyard_11022021_Mayor.csv,3,3,0.800911,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,317,385,1537,<NA>
357,Vineyard_11052019_CityCouncil.csv,7,7,0.237833,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,64,452,1089,<NA>
358,Westbrook_11052024_Mayor.csv,3,3,0.890167,"Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David",4464,5663,9560,<NA>
359,WoodlandHills_11022021_Mayor.csv,3,3,0.862595,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,128,262,655,<NA>


In [77]:
for i in range(len(trunc)):
    filename = trunc.at[i, "filename"]
    level = election_table.loc[election_table["filename"]==filename, "level"].values[0]
    trunc.at[i, "level"] = level

trunc

,filename,candidates,choices,gamma,og_irv,og_condorcet,og_plurality,Bvote_free_irv,Bvote_free_condorcet,Bvote_free_plurality,Tvote_free_irv,Tvote_free_condorcet,Tvote_free_plurality,bullet_votes,voluntarily_truncated_votes,total_votes,level
0,Alaska_04102020_PRESIDENTOFTHEUNITEDSTATES.csv,8,5,0.670398,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,Joseph R. Biden,<NA>,<NA>,<NA>,3922,<NA>,19742,FEDERAL
1,Alaska_08162022_HouseofRepresentativesSpecial.csv,3,4,0.966842,"Peltola, Mary S.","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.","Begich, Nick","Begich, Nick","Peltola, Mary S.",56054,133333,188852,FEDERAL
2,Alaska_11052024_President.csv,8,8,0.833383,Trump/Vance,Trump/Vance,Trump/Vance,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,Harris/Walz,228278,301747,338650,FEDERAL
3,Alaska_11052024_StateHouseD1.csv,3,4,0.953816,"Bynum, Jeremy T.","Bynum, Jeremy T.","Bynum, Jeremy T.","Moran, Agnes C.","Moran, Agnes C.","Moran, Agnes C.","Echohawk, Grant","Moran, Agnes C.","Bynum, Jeremy T.",5310,6769,8163,STATE
4,Alaska_11052024_StateHouseD15.csv,3,4,0.98254,"Costello, Mia","Costello, Mia","Costello, Mia","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny","Wells, Denny",6237,7576,8820,STATE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,Vineyard_11022021_Mayor.csv,3,3,0.800911,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,JULIE FULLMER,317,385,1537,LOCAL
357,Vineyard_11052019_CityCouncil.csv,7,7,0.237833,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,CRISTY WELSH,64,452,1089,LOCAL
358,Westbrook_11052024_Mayor.csv,3,3,0.890167,"Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David","Morse, David",4464,5663,9560,LOCAL
359,WoodlandHills_11022021_Mayor.csv,3,3,0.862595,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,BRENT WINDER,128,262,655,LOCAL


In [78]:
valid = (
    trunc["Tvote_free_irv"].notna()
    & trunc["Tvote_free_condorcet"].notna()
    & trunc["candidates"].notna()
)
print(valid.value_counts().get(True, 0))
dfv = trunc.loc[valid, ["filename", "gamma", "level", "candidates", "Tvote_free_irv", "Tvote_free_condorcet"]].copy()
dfv["candidates"] = dfv["candidates"].astype(int)
dfv["differs"] = (dfv["Tvote_free_irv"] != dfv["Tvote_free_condorcet"]).astype(int)
print(dfv[dfv["differs"]==1])

193
                                           filename     gamma  level  \
3                  Alaska_11052024_StateHouseD1.csv  0.953816  STATE   
17   Alaska_11082022_GovernorLieutenantGovernor.csv  0.880671  STATE   
70                    Burlington_03032009_Mayor.csv  0.651282  LOCAL   
85      LasCruces_11052019_MAYORCITYOFLASCRUCES.csv   0.49374  LOCAL   
303      SanFrancisco_11052019_DistrictAttorney.csv  0.751599  LOCAL   

     candidates      Tvote_free_irv   Tvote_free_condorcet  differs  
3             3     Echohawk, Grant        Moran, Agnes C.        1  
17            4  Dunleavy/Dahlstrom          Walker/Drygas        1  
70            5            Bob Kiss          Andy Montroll        1  
85           10       MIKE A TELLEZ  WILLIAM BILL MATTIACE        1  
303           4        CHESA BOUDIN                     -1        1  


In [80]:
valid = (
    trunc["Bvote_free_irv"].notna()
    & trunc["Bvote_free_condorcet"].notna()
    & trunc["candidates"].notna()
)
dfv = trunc.loc[valid, ["filename", "gamma", "level", "candidates", "Bvote_free_irv", "Bvote_free_condorcet"]].copy()
dfv["candidates"] = dfv["candidates"].astype(int)
dfv["differs"] = (dfv["Bvote_free_irv"] != dfv["Bvote_free_condorcet"]).astype(int)
print(dfv[dfv["differs"]==1])


                                              filename     gamma  level  \
17      Alaska_11082022_GovernorLieutenantGovernor.csv  0.880671  STATE   
22                 Alaska_11082022_HouseDistrict20.csv  0.853639  STATE   
70                       Burlington_03032009_Mayor.csv  0.651282  LOCAL   
108                     Minneapolis_11022021_Mayor.csv  0.883745  LOCAL   
211          NewYorkCity_06222021_DEMMayorCitywide.csv  0.462661  LOCAL   
260  PierceCounty_11042008_CountyAssessorTreasurer.csv  0.823079  LOCAL   
287                    SanFrancisco_11032015_Mayor.csv  0.845836  LOCAL   

     candidates      Bvote_free_irv Bvote_free_condorcet  differs  
17            4  Dunleavy/Dahlstrom        Walker/Drygas        1  
22            4     Gray, Andrew T.                   -1        1  
70            5            Bob Kiss        Andy Montroll        1  
108          18       Sheila Nezhad           Kate Knuth        1  
211          13   Kathryn A. Garcia                   -1   

In [55]:
import numpy as np
import pandas as pd

# Ensure numeric
for col in ["bullet_votes", "voluntarily_truncated_votes", "total_votes"]:
    trunc[col] = pd.to_numeric(trunc[col], errors="coerce")

# Valid denominator
mask = trunc["total_votes"] > 0

# Ratios
bullet_rate = trunc.loc[mask, "bullet_votes"] / trunc.loc[mask, "total_votes"]
trunc_rate  = trunc.loc[mask, "voluntarily_truncated_votes"] / trunc.loc[mask, "total_votes"]

def stats(s: pd.Series) -> dict:
    s = s.dropna()
    n = int(s.size)
    mean = s.mean()
    median = s.median()
    sd = s.std(ddof=1) if n > 1 else np.nan      # sample SD
    se = (sd / np.sqrt(n)) if n > 1 else np.nan  # standard error of mean
    return {"n": n, "mean": mean, "median": median, "sd": sd, "se": se}

summary = pd.DataFrame({
    "bullet_votes/total_votes": stats(bullet_rate),
    "voluntarily_truncated_votes/total_votes": stats(trunc_rate),
}).T

print(summary)


                                             n      mean    median        sd  \
bullet_votes/total_votes                 361.0  0.343033  0.320828  0.142326   
voluntarily_truncated_votes/total_votes  193.0  0.642663  0.657517  0.175420   

                                               se  
bullet_votes/total_votes                 0.007491  
voluntarily_truncated_votes/total_votes  0.012627  


In [82]:
trunc.to_csv("truncation_analysis.csv")

In [84]:
import numpy as np
import pandas as pd

# Ensure numeric
for col in ["bullet_votes", "voluntarily_truncated_votes", "total_votes"]:
    trunc[col] = pd.to_numeric(trunc[col], errors="coerce")

# Valid rows
mask = trunc["total_votes"] > 0
df_rates = trunc.loc[mask, ["level"]].copy()
df_rates["bullet_rate"] = trunc.loc[mask, "bullet_votes"] / trunc.loc[mask, "total_votes"]
df_rates["trunc_rate"]  = trunc.loc[mask, "voluntarily_truncated_votes"] / trunc.loc[mask, "total_votes"]

# Order levels nicely (optional)
lvl_order = pd.CategoricalDtype(categories=["FEDERAL", "STATE", "LOCAL"], ordered=True)
df_rates["level"] = df_rates["level"].astype("string").astype(lvl_order)

def summarize(series: pd.Series) -> pd.DataFrame:
    out = series.agg(['count', 'mean', 'median', 'std']).to_frame().T
    out.rename(columns={'count':'n', 'std':'sd'}, inplace=True)
    out['se'] = out['sd'] / np.sqrt(out['n']).where(out['n'] > 1)
    return out[['n','mean','median','sd','se']]

# --- BY LEVEL ---
br = df_rates.groupby("level", observed=True)["bullet_rate"].apply(summarize)
tr = df_rates.groupby("level", observed=True)["trunc_rate"].apply(summarize)

# Flatten indexes and add prefixes to distinguish the two sets
br.columns = [f"bullet_{c}" for c in br.columns]
tr.columns = [f"trunc_{c}"  for c in tr.columns]
by_level = pd.concat([br, tr], axis=1).sort_index()

print("By level (fractions in [0,1]):")
print(by_level)

# --- OVERALL ---
overall_br = summarize(df_rates["bullet_rate"]).add_prefix("bullet_")
overall_tr = summarize(df_rates["trunc_rate"]).add_prefix("trunc_")
overall = pd.concat([overall_br, overall_tr], axis=1)
print("\nOverall summary:")
print(overall)


By level (fractions in [0,1]):
                 bullet_n  bullet_mean  bullet_median  bullet_sd  bullet_se  \
level                                                                         
FEDERAL FEDERAL      15.0     0.391689       0.359669   0.141341   0.036494   
STATE   STATE        42.0     0.493826       0.504081   0.168986   0.026075   
LOCAL   LOCAL       304.0     0.319799       0.300491   0.124377   0.007134   

                 trunc_n  trunc_mean  trunc_median  trunc_sd  trunc_se  
level                                                                   
FEDERAL FEDERAL     10.0    0.752484      0.779219  0.124988  0.039525  
STATE   STATE       42.0    0.724038      0.792366  0.177588  0.027402  
LOCAL   LOCAL      141.0    0.610635      0.608946  0.167572  0.014112  

Overall summary:
             bullet_n  bullet_mean  bullet_median  bullet_sd  bullet_se  \
bullet_rate     361.0     0.343033       0.320828   0.142326   0.007491   
trunc_rate        NaN          NaN      